# Rendering-B independently annealed terminal endpoints — seed 1

Runs separate 3M, 10M, 30M, and 100M cosine schedules for each arm. Every endpoint restarts from its arm's original state and anneals to numerical zero. Attach the successful dense seed-1 output as a Kaggle input before running.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, urllib.request
REPO_URL = 'https://github.com/escher-bach/actuallybuildingstuff.git'
GIT_COMMIT = '__FINAL_COMMIT_SHA__'
CONFIG_REL = 'step1/configs/kaggle/t4x2_rendering_b_terminal_seed1.toml'
WORKING = Path('/kaggle/working')
SOURCE = WORKING / 'actuallybuildingstuff'
PROJECT = SOURCE / 'baby-llm-foundations'
OUTPUT = WORKING / 'rendering-b-terminal-seed1'
assert len(GIT_COMMIT) == 40 and all(c in '0123456789abcdef' for c in GIT_COMMIT)
assert not SOURCE.exists(), f'fresh batch session required: {SOURCE}'

In [ ]:
env = os.environ.copy()
env.update({'GIT_TERMINAL_PROMPT':'0','PYTHONUNBUFFERED':'1','PIP_DISABLE_PIP_VERSION_CHECK':'1','WANDB_MODE':'disabled','TOKENIZERS_PARALLELISM':'false'})
subprocess.run(['git','clone',REPO_URL,str(SOURCE)],check=True,env=env)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach',GIT_COMMIT],check=True,env=env)
assert subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True,env=env).strip() == GIT_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements-kaggle.txt')],check=True,env=env)
if not shutil.which('cargo'):
    rustup=WORKING/'rustup-init'; urllib.request.urlretrieve('https://static.rust-lang.org/rustup/dist/x86_64-unknown-linux-gnu/rustup-init',rustup); rustup.chmod(0o755)
    subprocess.run([str(rustup),'-y','--profile','minimal','--default-toolchain','1.85.0'],check=True,env=env); env['PATH']=str(Path.home()/'.cargo/bin')+os.pathsep+env['PATH']
subprocess.run([sys.executable,'-m','maturin','build','--release','--manifest-path',str(PROJECT/'step1/crates/world-py/Cargo.toml')],cwd=str(PROJECT/'step1'),check=True,env=env)
wheel=sorted((PROJECT/'step1/target/wheels').glob('world_py-*.whl'))[-1]
subprocess.run([sys.executable,'-m','pip','install','--force-reinstall',str(wheel)],check=True,env=env)

In [ ]:
sys.path.insert(0,str(PROJECT/'step1/python'))
from step1_experiments.transfer import load_transfer_config, locate_dense_source
config = load_transfer_config(PROJECT/CONFIG_REL)
DENSE_SOURCE = locate_dense_source([Path('/kaggle/input')], config['source'])
print('Using exact dense source:',DENSE_SOURCE)

In [ ]:
cmd=['torchrun','--standalone','--nproc_per_node=2','-m','step1_experiments.terminal_transfer','--config',str(PROJECT/CONFIG_REL),'--source-run',str(DENSE_SOURCE),'--output-dir',str(OUTPUT)]
subprocess.run(cmd,cwd=str(PROJECT/'step1/python'),env=env,check=True)

In [ ]:
import json, math
report=json.loads((OUTPUT/'rendering_b_terminal_transfer_report.json').read_text())
budgets=[0,92,306,916,3052]; tokens=[step*32768 for step in budgets]
assert report['contract']=='step1_rendering_b_terminal_transfer_v1'
assert report['plan']['budgets_updates']==budgets and report['plan']['budgets_nominal_global_input_tokens']==tokens
assert report['schedule_policy']['restart_from_arm_initialization_per_budget'] is True
for arm in report['arms'].values():
    assert [p['budget_updates'] for p in arm['endpoints']]==budgets
    for p in arm['endpoints'][1:]:
        assert p['schedule']['max_steps']==p['budget_updates'] and p['schedule']['terminal_learning_rate']<=1e-9
        assert p['serialization']['exact'] is True
        assert all(v is None or math.isfinite(v) for metrics in p['metrics'].values() for v in metrics.values())
print(json.dumps({'paired_diagnostics':report['paired_diagnostics'],'arms':{k:v['endpoints'] for k,v in report['arms'].items()}},indent=2))